In [1]:
# The Random Forest is a type of "ensemble" algorithm, meaning that it combines many smaller algorithms to make better predictions.
# It uses a very simple kind of machine learning algorithm called a decision tree. 
# A decision tree makes predictions by examining the values of features in the input. 
# Like a flow chart with IF statements. Decision trees are very quick and simple, but they tend to overfit.
# In our case, the "features" are the elements of the Vector - in other words, it's the number of times that a particular 
# word appears in the product description. So you can think of it something like this:
"""
Decision Tree
- IF the word "TV" appears more than 3 times THEN
-- IF the word "LED" appears more than 2 times THEN
--- IF the word "HD" appears at least once THEN
---- Price = $500
With Random Forest, multiple decision trees are created. Each one is trained with a different random subset of the data, 
and a different random subset of the features. You can see above that we specify 100 trees, which is the default.
Then the Random Forest model simply takes the average of all its trees to product the final result.
"""

import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

LITE_MODE = True
username = "allanhadoop"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [3]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [5]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)

np.random.seed(42)
# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']
subset = 5_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [6]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
evaluate(random_forest, test)
# With random forest error is further reduced to $73.33 

  0%|          | 0/200 [00:00<?, ?it/s]

$117 $33 $5 $26 $71 $105 $55 $55 $130 $69 $655 $285 $36 $74 $2 $121 $88 $99 $67 $1 $27 $5 $6 $30 $272 $280 $86 $43 $10 $47 $35 $60 $37 $16 $98 $457 $147 $36 $102 $76 $142 $53 $10 $54 $126 $53 $70 $38 $50 $3 $12 $23 $189 $31 $106 $19 $41 $124 $19 $42 $94 $9 $49 $22 $415 $1 $31 $257 $13 $115 $24 $34 $81 $160 $10 $151 $119 $40 $35 $45 $16 $55 $21 $51 $54 $25 $16 $121 $14 $218 $14 $86 $1 $4 $27 $103 $7 $28 $199 $246 $36 $8 $27 $72 $39 $38 $92 $215 $33 $174 $30 $59 $111 $94 $113 $19 $128 $22 $32 $37 $34 $23 $68 $36 $22 $15 $107 $55 $96 $12 $50 $26 $88 $13 $81 $34 $14 $10 $42 $38 $10 $126 $20 $210 $46 $85 $53 $309 $59 $2 $23 $121 $53 $88 $55 $105 $143 $73 $44 $7 $48 $14 $17 $38 $262 $33 $262 $29 $49 $38 $57 $11 $260 $18 $121 $35 $36 $31 $55 $20 $435 $55 $37 $2 $73 $83 $50 $66 $18 $18 $21 $66 $45 $29 $3 $25 $4 $97 $38 $10 

In [ ]:
# Like Random Forest, XGBoost is also an ensemble model that combines multiple decision trees.
# But unlike Random Forest, XGBoost builds one tree after another, with each next tree correcting for errors in the prior trees, 
# using 'gradient descent'. It's much faster than Random Forest, so we can run it for the full dataset, and it's typically better at generalizing.


import xgboost as xgb
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [9]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [10]:
evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$82 $81 $30 $16 $107 $160 $39 $3 $16 $21 $601 $245 $107 $107 $0 $17 $55 $13 $15 $15 $11 $25 $5 $68 $234 $321 $95 $31 $40 $28 $84 $55 $9 $9 $54 $124 $24 $69 $164 $68 $154 $88 $63 $8 $118 $113 $37 $57 $43 $8 $30 $37 $126 $8 $236 $95 $61 $193 $2 $41 $72 $8 $71 $35 $444 $33 $46 $208 $50 $100 $2 $43 $92 $106 $13 $200 $222 $10 $34 $117 $22 $95 $87 $40 $19 $50 $63 $124 $26 $226 $34 $196 $27 $3 $25 $93 $124 $26 $199 $168 $114 $40 $23 $61 $5 $91 $175 $207 $31 $66 $39 $64 $72 $54 $79 $140 $129 $29 $136 $42 $45 $98 $35 $33 $135 $4 $16 $97 $179 $99 $38 $52 $141 $35 $9 $70 $74 $48 $164 $93 $2 $148 $55 $89 $82 $96 $36 $260 $11 $5 $19 $163 $46 $60 $25 $96 $117 $16 $21 $4 $74 $21 $8 $13 $397 $2 $147 $50 $15 $11 $22 $26 $209 $32 $31 $47 $50 $32 $37 $58 $423 $12 $137 $30 $113 $59 $61 $45 $65 $30 $47 $20 $16 $17 $4 $71 $57 $41 $40 $26 